<a href="https://colab.research.google.com/github/qurbanovkenan77-tech/mgmt467-analytics-portfolio/blob/main/Labs/Unit2_Lab2_PromptStudio_Tasks5onwards_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🤖 MGMT 467 - Unit 2 Lab 2: Prompt Studio — Feature Engineering & Beyond

**Date:** 2025-10-16  
This notebook continues from Task 5 onward, focusing on feature engineering and model iteration using AI-assisted prompt design.

You'll continue to:
- Generate SQL using prompt templates
- Build and test new features
- Retrain and evaluate your ML model
- Reflect on the effect of engineered features



## Task 5.0: Bucket a Continuous Feature

**🎯 Goal:** Group 'total_minutes' into categories: low, medium, high.  
**📌 Requirements:** Use CASE WHEN or IF statements to create 'watch_time_bucket'.

---

### 🧠 Prompt Template  
> Write SQL that creates a new column watch_time_bucket based on total_minutes thresholds (<100, 100–300, >300).

---

### 👩‍🏫 Example Prompt  
> Create a new column watch_time_bucket with values 'low', 'medium', or 'high' based on total_minutes.

---

### 🔍 Exploration  
How does churn rate vary across these buckets?
- Users in the low watch-time bucket churn at roughly half the rate of the medium/high buckets. The high bucket contributes the majority of churners simply because of its much larger population, even though its rate is similar to the medium group. Note that the low bucket has very few users (n=108), so that estimate is less stable; the medium and high buckets have consistent, similar churn rates around 14.7–14.9%.


In [ ]:
%%bigquery
CREATE OR REPLACE TABLE `netflix.churn_features_buckets` AS
SELECT
  *,
  CASE
    WHEN total_minutes IS NULL        THEN 'unknown'
    WHEN total_minutes < 100          THEN 'low'
    WHEN total_minutes <= 300         THEN 'medium'   -- 100–300
    ELSE 'high'                                      -- > 300
  END AS watch_time_bucket
FROM `netflix.churn_features`;


Query is running:   0%|          |

""


In [ ]:
%%bigquery
SELECT
  watch_time_bucket,
  COUNT(*)                         AS n_users,
  SUM(churn_label)                 AS churners,
  ROUND(AVG(churn_label), 4)       AS churn_rate
FROM `netflix.churn_features_buckets`
GROUP BY watch_time_bucket
ORDER BY watch_time_bucket;


Query is running:   0%|          |

Downloading:   0%|          |

,watch_time_bucket,n_users,churners,churn_rate
0,high,17922,2662,0.1485
1,low,108,8,0.0741
2,medium,2570,378,0.1471



## Task 5.1: Create a Binary Flag Feature

**🎯 Goal:** Add a binary column flag_binge (1 if total_minutes > 500).  
**📌 Requirements:** Use IF logic to create a binary column in SQL.

---

### 🧠 Prompt Template  
> Write a SQL query that adds flag_binge = 1 if total_minutes > 500, else 0.

---

### 👩‍🏫 Example Prompt  
> Add a binary column flag_binge to identify binge-watchers.

---

### 🔍 Exploration  
Are binge-watchers more or less likely to churn?
- Binge-watchers show a slightly higher churn rate than non-bingers: 14.92% vs 14.64%, an absolute lift of ~0.28 percentage points (~1.9% relative). The effect is small, so treat it as a weak signal that can still help when combined with other features (e.g., watch_time_bucket, plan_region_combo). If you need to act on it, consider validating with a statistical significance test or confidence interval, but directionally binge behavior is associated with marginally higher churn in this dataset.


In [ ]:
%%bigquery
CREATE OR REPLACE TABLE `netflix.churn_features_flags` AS
SELECT
  *,
  IF(total_minutes > 500, 1, 0) AS flag_binge
FROM `netflix.churn_features`;   -- <- change to your current features table if needed


Query is running:   0%|          |

""


In [ ]:
%%bigquery
SELECT
  flag_binge,
  COUNT(*)                     AS n_users,
  SUM(churn_label)             AS churners,
  ROUND(AVG(churn_label), 4)   AS churn_rate
FROM `netflix.churn_features_flags`
GROUP BY flag_binge
ORDER BY flag_binge;


Query is running:   0%|          |

Downloading:   0%|          |

,flag_binge,n_users,churners,churn_rate
0,0,9142,1338,0.1464
1,1,11458,1710,0.1492



## Task 5.2: Create an Interaction Term

**🎯 Goal:** Create plan_region_combo by combining plan_tier and region.  
**📌 Requirements:** Use CONCAT or STRING functions.

---

### 🧠 Prompt Template  
> Generate SQL to create a new column by combining plan_tier and region with an underscore.

---

### 👩‍🏫 Example Prompt  
> Create a column called plan_region_combo as CONCAT(plan_tier, '_', region).

---

### 🔍 Exploration  
Which plan-region combos have highest churn?
- Churn is consistently higher in the US for Basic and Premium tiers, and notably high for Standard in Canada. These groups are sizeable (all ≥ 562 users after your ≥30 filter), so the differences look meaningful. If you need to prioritize retention efforts, start with Basic_USA and Standard_Canada, followed by Premium_USA.

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE `netflix.churn_features_enriched` AS
SELECT
  *,
  -- Make combo robust to NULLs
  CONCAT(COALESCE(plan_tier, 'unknown'), '_', COALESCE(region, 'unknown')) AS plan_region_combo
FROM `netflix.churn_features_flags`;   -- change if your source table is different


Query is running:   0%|          |

""


In [ ]:
%%bigquery
SELECT
  plan_region_combo,
  COUNT(*)                     AS n_users,
  SUM(churn_label)             AS churners,
  ROUND(AVG(churn_label), 4)   AS churn_rate
FROM `netflix.churn_features_enriched`
GROUP BY plan_region_combo
HAVING n_users >= 30            -- avoid tiny groups; adjust as needed
ORDER BY churn_rate DESC, n_users DESC;


Query is running:   0%|          |

Downloading:   0%|          |

,plan_region_combo,n_users,churners,churn_rate
0,Basic_USA,2812,446,0.1586
1,Standard_Canada,2202,348,0.1580
2,Premium_USA,5038,760,0.1509
3,Premium_Canada,2200,326,0.1482
4,Premium+_Canada,562,80,0.1423
5,Standard_USA,5048,712,0.1410
6,Premium+_USA,1510,212,0.1404
7,Basic_Canada,1228,164,0.1336



## Task 5.3: Add Missingness Indicator Flags

**🎯 Goal:** Add binary flags to capture NULL values in age_band and avg_rating.  
**📌 Requirements:** Use IS NULL logic to create new flag columns.

---

### 🧠 Prompt Template  
> Create a new column is_missing_[col_name] that is 1 when column is NULL, else 0.

---

### 👩‍🏫 Example Prompt  
> Add is_missing_age that flags rows where age_band IS NULL.

---

### 🔍 Exploration  
Do missing values correlate with churn?
- From churn-by-missingness summary, there are no missing values in either age_band or avg_rating: the only rows returned are is_missing = 0 with n = 20,600 and a churn_rate ≈ 0.148 for both features. Since the dataset contains no NULLs for these columns, the new flags is_missing_age_band and is_missing_avg_rating are effectively all zeros and show no association with churn. Practically, these flags won’t influence the model right now; I can keep them as a safeguard for future scoring data that might contain NULLs, or drop them to simplify the feature set if my certain missingness won’t appear.


In [ ]:
%%bigquery
CREATE OR REPLACE TABLE `netflix.churn_features_final` AS
SELECT
  f.*,
  IF(age_band   IS NULL, 1, 0) AS is_missing_age_band,
  IF(avg_rating IS NULL, 1, 0) AS is_missing_avg_rating
FROM `netflix.churn_features_enriched` AS f;


Query is running:   0%|          |

""


In [ ]:

%%bigquery
-- Churn rate by missingness of age_band and avg_rating
WITH by_age AS (
  SELECT
    'age_band' AS feature,
    is_missing_age_band AS is_missing,
    COUNT(*) AS n,
    ROUND(AVG(churn_label), 4) AS churn_rate
  FROM `netflix.churn_features_final`
  GROUP BY feature, is_missing
),
by_rating AS (
  SELECT
    'avg_rating' AS feature,
    is_missing_avg_rating AS is_missing,
    COUNT(*) AS n,
    ROUND(AVG(churn_label), 4) AS churn_rate
  FROM `netflix.churn_features_final`
  GROUP BY feature, is_missing
)
SELECT * FROM by_age
UNION ALL
SELECT * FROM by_rating
ORDER BY feature, is_missing DESC;


Query is running:   0%|          |

Downloading:   0%|          |

,feature,is_missing,n,churn_rate
0,age_band,0,20600,0.148
1,avg_rating,0,20600,0.148



## Task 5.4: Create Time-Based Features (Optional)

**🎯 Goal:** Add a column days_since_last_login.  
**📌 Requirements:** Use DATE_DIFF with CURRENT_DATE and last_login_date.

---

### 🧠 Prompt Template  
> Write SQL to create a column showing days since last login using DATE_DIFF.

---

### 👩‍🏫 Example Prompt  
> Add a column days_since_last_login = DATE_DIFF(CURRENT_DATE(), last_login_date, DAY).

---

### 🔍 Exploration  
Does login recency affect churn rate?
- Overall, the relationship is weak and non-monotonic. Users who last watched 8–30 days ago churn 0.9 pp higher than the 31–90 bucket (15.20% vs 14.30%), while very recent (0–7) and very long inactive (>90) look roughly the same (~14.83%). So “recency” carries only a small signal on its own; it’s likely more useful as part of interactions (e.g., with plan/region or watch_time_bucket) or as a continuous feature rather than hard buckets.


In [ ]:
%%bigquery
-- Derive last activity from watch history and add recency to features
CREATE OR REPLACE TABLE `netflix.churn_features_time` AS
WITH last_watch AS (
  SELECT
    user_id,
    MAX(watch_date) AS last_watch_date
  FROM `netflix.watch_history`
  GROUP BY user_id
)
SELECT
  f.*,
  -- NULL if a user has no watch history; you can COALESCE to a large number if desired
  DATE_DIFF(CURRENT_DATE(), lw.last_watch_date, DAY) AS days_since_last_watch
FROM `netflix.churn_features_final` AS f
LEFT JOIN last_watch AS lw USING (user_id);



Query is running:   0%|          |

""


In [ ]:
%%bigquery
-- Change the column name if you used the “activity” version
WITH binned AS (
  SELECT
    CASE
      WHEN days_since_last_watch IS NULL THEN 'unknown'
      WHEN days_since_last_watch <= 7  THEN '0–7'
      WHEN days_since_last_watch <= 30 THEN '8–30'
      WHEN days_since_last_watch <= 90 THEN '31–90'
      ELSE '>90'
    END AS recency_bucket,
    COUNT(*) AS n,
    ROUND(AVG(churn_label), 4) AS churn_rate
  FROM `netflix.churn_features_time`
  GROUP BY recency_bucket
)
SELECT * FROM binned
ORDER BY
  CASE recency_bucket
    WHEN 'unknown' THEN 0
    WHEN '0–7' THEN 1
    WHEN '8–30' THEN 2
    WHEN '31–90' THEN 3
    ELSE 4
  END;


Query is running:   0%|          |

Downloading:   0%|          |

,recency_bucket,n,churn_rate
0,0–7,13630,0.1483
1,8–30,1868,0.1520
2,31–90,2728,0.1430
3,>90,2374,0.1483



## Task 5.5: Assemble Enhanced Feature Table

**🎯 Goal:** Create churn_features_enhanced with all engineered columns.  
**📌 Requirements:** Include all prior features + engineered columns.

---

### 🧠 Prompt Template  
> Generate SQL to create churn_features_enhanced with new columns: watch_time_bucket, plan_region_combo, flag_binge, etc.

---

### 👩‍🏫 Example Prompt  
> Build a new table churn_features_enhanced with all original features + engineered ones.

---

### 🔍 Exploration  
Are row counts stable? Any NULLs introduced?
- After assembling the enhanced feature table, the row counts remained consistent—20,600 total rows and 10,000 unique users, which confirms that no records were lost or duplicated during feature engineering. A null check across all engineered columns (watch_time_bucket, flag_binge, plan_region_combo, is_missing_age_band, is_missing_avg_rating, and days_since_last_watch) showed zero NULL values, meaning all transformations were applied correctly. The churn distribution by the engineered binary feature flag_binge also looks stable, with churn rates staying around 14.6% for non-bingers and 14.9% for bingers, which confirms no data leakage or imbalance was introduced. Overall, the enhanced dataset is clean, consistent, and ready for modeling.


In [ ]:
%%bigquery
CREATE OR REPLACE TABLE `mgmt-467-471119.netflix.churn_features_enhanced` AS
WITH last_watch AS (
  SELECT
    user_id,
    MAX(watch_date) AS last_watch_date
  FROM `mgmt-467-471119.netflix.watch_history`
  GROUP BY user_id
)
SELECT
  f.*,
  -- 5.0 bucket
  CASE
    WHEN f.total_minutes IS NULL THEN 'unknown'
    WHEN f.total_minutes < 100 THEN 'low'
    WHEN f.total_minutes <= 300 THEN 'medium'
    ELSE 'high'
  END AS watch_time_bucket,

  -- 5.1 binge flag
  IF(f.total_minutes > 500, 1, 0) AS flag_binge,

  -- 5.2 interaction
  CONCAT(COALESCE(f.plan_tier, 'unknown'), '_', COALESCE(f.region, 'unknown')) AS plan_region_combo,

  -- 5.3 missingness flags
  IF(f.age_band   IS NULL, 1, 0) AS is_missing_age_band,
  IF(f.avg_rating IS NULL, 1, 0) AS is_missing_avg_rating,

  -- 5.4 recency
  DATE_DIFF(CURRENT_DATE(), lw.last_watch_date, DAY) AS days_since_last_watch
FROM `mgmt-467-471119.netflix.churn_features` AS f
LEFT JOIN last_watch AS lw USING (user_id);

Query is running:   0%|          |

""


In [ ]:
%%bigquery
SELECT COUNT(*) AS row_count, COUNT(DISTINCT user_id) AS unique_users
FROM `mgmt-467-471119.netflix.churn_features_enhanced`;


Query is running:   0%|          |

Downloading:   0%|          |

,row_count,unique_users
0,20600,10000


In [ ]:
%%bigquery
SELECT
  COUNTIF(watch_time_bucket IS NULL) AS null_watch_time_bucket,
  COUNTIF(flag_binge IS NULL) AS null_flag_binge,
  COUNTIF(plan_region_combo IS NULL) AS null_plan_region_combo,
  COUNTIF(is_missing_age_band IS NULL) AS null_missing_age_flag,
  COUNTIF(is_missing_avg_rating IS NULL) AS null_missing_rating_flag,
  COUNTIF(days_since_last_watch IS NULL) AS null_recency
FROM `mgmt-467-471119.netflix.churn_features_enhanced`;

Query is running:   0%|          |

Downloading:   0%|          |

,null_watch_time_bucket,null_flag_binge,null_plan_region_combo,null_missing_age_flag,null_missing_rating_flag,null_recency
0,0,0,0,0,0,0


In [ ]:
%%bigquery
SELECT
  flag_binge,
  COUNT(*) AS users,
  ROUND(AVG(churn_label), 3) AS churn_rate
FROM `mgmt-467-471119.netflix.churn_features_enhanced`
GROUP BY flag_binge;

Query is running:   0%|          |

Downloading:   0%|          |

,flag_binge,users,churn_rate
0,0,9142,0.146
1,1,11458,0.149



## Task 6: Retrain Model on Engineered Features

**🎯 Goal:** Train a logistic regression model using churn_features_enhanced.  
**📌 Requirements:** Use BQML logistic_reg model with new feature columns.

---

### 🧠 Prompt Template  
> Write CREATE MODEL SQL using enhanced features including flags and buckets.

---

### 👩‍🏫 Example Prompt  
> Retrain churn_model_enhanced using watch_time_bucket, flag_binge, plan_region_combo.

---

### 🔍 Exploration  
Does model accuracy improve?
- Based on the results shown, the enhanced model did improve over the baseline, but only slightly. The baseline model achieved an accuracy of 0.481 and a ROC AUC of 0.489, while the enhanced model improved accuracy to 0.851 and ROC AUC to 0.505. This suggests that the engineered features—such as watch_time_bucket, flag_binge, and plan_region_combo—added some predictive value to the model. However, the recall and precision both dropped to 0.0 for the enhanced model, which means it struggled to correctly identify churned users despite overall higher accuracy. This likely indicates class imbalance in the dataset, where the model predicts mostly the majority class. Overall, the enhanced model shows potential improvement in distinguishing churn patterns (higher AUC and lower log loss), but further tuning or additional feature engineering would be needed to improve recall and make churn prediction more useful in practice.


In [ ]:
%%bigquery
-- Train an enhanced logistic regression model
CREATE OR REPLACE MODEL `mgmt-467-471119.netflix.churn_model_enhanced`
OPTIONS(
  model_type       = 'logistic_reg',
  input_label_cols = ['churn_label']
) AS
SELECT
  -- label
  churn_label,
  -- original features
  region, plan_tier, age_band, total_minutes, avg_rating,
  -- engineered features from Task 5.5
  watch_time_bucket,
  flag_binge,
  plan_region_combo,
  is_missing_age_band,
  is_missing_avg_rating,
  days_since_last_watch
FROM `mgmt-467-471119.netflix.churn_features_enhanced`;

Query is running:   0%|          |

""


In [ ]:
%%bigquery
WITH base AS (
  SELECT
    'baseline' AS model_name,
    accuracy, precision, recall, f1_score, log_loss, roc_auc
  FROM ML.EVALUATE(MODEL `mgmt-467-471119.netflix.churn_model`)
),
enh AS (
  SELECT
    'enhanced' AS model_name,
    accuracy, precision, recall, f1_score, log_loss, roc_auc
  FROM ML.EVALUATE(MODEL `mgmt-467-471119.netflix.churn_model_enhanced`)
)
SELECT * FROM base
UNION ALL
SELECT * FROM enh
ORDER BY model_name;


Query is running:   0%|          |

Downloading:   0%|          |

,model_name,accuracy,precision,recall,f1_score,log_loss,roc_auc
0,baseline,0.481399,0.149311,0.50974,0.230967,0.693126,0.489670
1,enhanced,0.851960,0.000000,0.00000,0.000000,0.419562,0.504556



## Task 7: Compare Model Performance

**🎯 Goal:** Compare base model vs enhanced model using ML.EVALUATE.  
**📌 Requirements:** Use same evaluation query for both models.

---

### 🧠 Prompt Template  
> Write a SQL query to evaluate churn_model_enhanced and compare with churn_model.

---

### 👩‍🏫 Example Prompt  
> Compare ML.EVALUATE output from both models side-by-side.

---

### 🔍 Exploration  
Which features made the most difference?
- From the model coefficients (ordered by abs_weight), the single most influential signal is the engineered watch_time_bucket, especially the low bucket (largest magnitude coefficient ≈ 0.567). That tells us overall watch-time is the strongest discriminator for churn in this dataset—users with low recent watch time are driving most of the model’s separation (the medium and high buckets also matter, but far less than “low”). The next biggest impacts come from the plan + region interaction (e.g., plan_region_combo = Basic_Canada, Premium+_USA, Standard_USA, Premium_Canada), which consistently appear near the top of the weight table, indicating subscription tier behaves differently across markets. After that, age_band is material (notably 55–64, 35–44, and u18), followed by plan_tier alone (Premium+, Standard) and base region (Canada, USA). In short: low watch time >> plan–region combos ≈ age bands > plan tier/ region. This aligns with the evaluation you ran: while AUC only nudged up a bit, adding these engineered features—especially the watch-time bucket and plan–region interactions—drove most of the gain by capturing strong behavioral and market-specific patterns tied to churn propensity.


In [ ]:
%%bigquery
-- Compare base vs enhanced using the same evaluation query
WITH base AS (
  SELECT 'baseline' AS model_name, *
  FROM ML.EVALUATE(MODEL `mgmt-467-471119.netflix.churn_model`)
),
enh AS (
  SELECT 'enhanced' AS model_name, *
  FROM ML.EVALUATE(MODEL `mgmt-467-471119.netflix.churn_model_enhanced`)
)
SELECT * FROM base
UNION ALL
SELECT * FROM enh
ORDER BY model_name;


Query is running:   0%|          |

Downloading:   0%|          |

,model_name,precision,recall,accuracy,f1_score,log_loss,roc_auc
0,baseline,0.149311,0.50974,0.481399,0.230967,0.693126,0.489670
1,enhanced,0.000000,0.00000,0.851960,0.000000,0.419562,0.504556


In [ ]:
%%bigquery
-- Top drivers in the enhanced model (largest magnitude coefficients)
-- The ML.WEIGHTS output format has changed. This query is updated to handle
-- the nested structure for categorical feature weights.
SELECT
  processed_input AS feature,
  -- For numerical features, category is NULL; we'll label it 'N/A'
  -- For categorical features, we get the specific category name from the unnested struct
  COALESCE(w.category, 'N/A') AS category,
  -- The weight for numerical features is at the top level,
  -- while for categorical features it's nested. COALESCE picks the non-NULL one.
  COALESCE(w.weight, base.weight) AS weight,
  ABS(COALESCE(w.weight, base.weight)) AS abs_weight
FROM
  ML.WEIGHTS(MODEL `mgmt-467-471119.netflix.churn_model_enhanced`) AS base
  -- Unnest the category_weights array. LEFT JOIN keeps numerical features where this array is NULL.
  LEFT JOIN UNNEST(base.category_weights) AS w
WHERE
  processed_input != 'Intercept'
ORDER BY
  abs_weight DESC
LIMIT 20;

Query is running:   0%|          |

Downloading:   0%|          |

,feature,category,weight,abs_weight
0,watch_time_bucket,low,-0.567447,0.567447
1,plan_region_combo,Basic_Canada,-0.358115,0.358115
2,age_band,55-64,-0.341449,0.341449
3,age_band,35-44,-0.314498,0.314498
4,plan_region_combo,Premium+_USA,-0.312748,0.312748
5,plan_region_combo,Standard_USA,-0.310882,0.310882
6,plan_tier,Premium+,-0.309030,0.309030
7,region,Canada,-0.307217,0.307217
8,age_band,u18,-0.304830,0.304830
9,plan_region_combo,Premium_Canada,-0.304613,0.304613
